## 평가 데이터셋(같은 출원인, 다른 출원번호) 기반 벡터DB 검색 성능 평가
- **HitRate@K**: Top-K 안에 relevant_image가 포함된 비율 (K = 5, 10, 20, 100)

In [ ]:
import csv
import os
from collections import defaultdict
import chromadb
import pandas as pd

# 경로 설정
GT_CSV    = r"C:\Users\playdata2\Desktop\SKN_AI_20\SKN20-FINAL-2TEAM(vb1)\design\ground_truth\ground_truth_task2_filtered.csv"
CHROMA_DB = r"C:\Users\playdata2\Desktop\SKN_AI_20\SKN20-FINAL-2TEAM(vb1)\design\chroma_db"
IMAGE_DIR = r"C:\Users\playdata2\Desktop\SKN_AI_20\SKN20-FINAL-2TEAM(vb1)\design\data\images(21,895개)"

K_LIST = [5, 10, 20, 100]
TOP_K  = max(K_LIST)

In [10]:
# 이미지 파일 목록을 미리 로드
image_files = set(os.listdir(IMAGE_DIR))
print(f"이미지 파일 수: {len(image_files)}")

def image_filename_to_db_id(filename):
    stem = os.path.splitext(filename)[0]
    last_us = stem.rfind('_')
    design_id = stem[:last_us]
    img_num = stem[last_us + 1:]
    return f"{design_id}-IMG-{int(img_num)}"


def db_id_to_image_filename(db_id):
    """'3020250000208-09-01-0-IMG-0' -> '3020250000208-09-01-0_000.jpg'"""
    parts = db_id.split("-IMG-")
    if len(parts) != 2:
        return None
    design_id = parts[0]
    prefix = f"{design_id}_"
    for fname in image_files:
        if fname.startswith(prefix):
            return fname
    return None


이미지 파일 수: 20778


### 1. 평가 데이터셋 로드

In [3]:
with open(GT_CSV, encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    gt_rows = list(reader)

# query_image별 relevant_image 그룹핑
query_relevants = defaultdict(set)
query_types = {}
for row in gt_rows:
    q = row["query_image"]
    r = row["relevant_image"]
    query_relevants[q].add(r)
    query_types[q] = row["similarity_type"]

print(f"Ground Truth 로드: {len(gt_rows)}쌍, 고유 query: {len(query_relevants)}개")

Ground Truth 로드: 16쌍, 고유 query: 16개


### 2. ChromaDB 연결 & ID 매핑 구축

In [7]:
client = chromadb.PersistentClient(path=CHROMA_DB)
collection = client.get_collection(name="design")
print(f"ChromaDB 컬렉션 '{collection.name}' 로드: {collection.count()}개 벡터")

# DB ID -> 이미지 파일명 역매핑
all_ids = collection.get(include=[])["ids"]
dbid_to_filename = {}
for db_id in all_ids:
    fname = db_id_to_image_filename(db_id)
    if fname:
        dbid_to_filename[db_id] = fname

print(f"DB ID -> 파일명 매핑: {len(dbid_to_filename)}개")

ChromaDB 컬렉션 'design' 로드: 21801개 벡터
DB ID -> 파일명 매핑: 20681개


### 3. 쿼리별 검색 & 평가

In [11]:
# 디버깅: 첫 3개 쿼리의 변환 결과 vs 실제 DB ID 비교
sample_queries = list(query_relevants.keys())[:3]

for q in sample_queries:
    db_id = image_filename_to_db_id(q)
    print(f"파일명: {q}")
    print(f"변환된 DB ID: {db_id}")
    
    # 이 ID가 DB에 있는지 확인
    try:
        result = collection.get(ids=[db_id], include=[])
        print(f"DB 조회: {'있음' if result['ids'] else '없음'}")
    except Exception as e:
        print(f"DB 조회 실패: {e}")
    
    print()

# 실제 DB에 저장된 ID 샘플 5개 출력
print("=" * 50)
print("실제 DB ID 샘플 5개:")
for db_id in all_ids[:5]:
    print(f"  {db_id}")


파일명: 3020250019104-api_xml-0_000.jpg
변환된 DB ID: 3020250019104-api_xml-0-IMG-0
DB 조회: 있음

파일명: 3020170036338-api_xml-0_000.jpg
변환된 DB ID: 3020170036338-api_xml-0-IMG-0
DB 조회: 있음

파일명: 3020170010907-api_xml-0_000.jpg
변환된 DB ID: 3020170010907-api_xml-0-IMG-0
DB 조회: 있음

실제 DB ID 샘플 5개:
  3020180025466-api_xml-1-IMG-1
  3020120015713-api_xml-1-IMG-1
  3020220050756-api_xml-0-IMG-0
  3020210026507-api_xml-2-IMG-2
  3020190011820-api_xml-2-IMG-2


In [ ]:
results_table = []
hit_sums = {k: 0 for k in K_LIST}
total_queries = 0
skipped = 0

for query_img, relevant_set in query_relevants.items():
    query_db_id = image_filename_to_db_id(query_img)

    try:
        query_data = collection.get(ids=[query_db_id], include=["embeddings"])
    except Exception:
        skipped += 1
        continue

    if query_data["embeddings"] is None or len(query_data["embeddings"]) == 0:
        skipped += 1
        continue

    query_embedding = query_data["embeddings"][0]

    # TOP_K+1 검색 후 자기 자신 제외
    search_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=TOP_K + 1,
    )

    retrieved_ids = [
        rid for rid in search_results["ids"][0] if rid != query_db_id
    ][:TOP_K]

    retrieved_distances = []
    for i, rid in enumerate(search_results["ids"][0]):
        if rid != query_db_id and len(retrieved_distances) < TOP_K:
            retrieved_distances.append(search_results["distances"][0][i])

    retrieved_filenames = [dbid_to_filename.get(rid, rid) for rid in retrieved_ids]

    # HitRate@K: relevant이 top-K 안에 있으면 hit
    first_hit_rank = None
    for rank, fname in enumerate(retrieved_filenames, start=1):
        if fname in relevant_set:
            first_hit_rank = rank
            break

    hits_at_k = {}
    for k in K_LIST:
        hit = int(first_hit_rank is not None and first_hit_rank <= k)
        hits_at_k[k] = hit
        hit_sums[k] += hit

    total_queries += 1

    results_table.append({
        "query":           query_img,
        "type":            query_types[query_img],
        "n_relevant":      len(relevant_set),
        "first_hit_rank":  first_hit_rank if first_hit_rank is not None else "miss",
        **{f"hit@{k}": hits_at_k[k] for k in K_LIST},
        "top_retrieved":   retrieved_filenames[:5],
        "top_dist":        retrieved_distances[:5],
    })

print(f"평가 완료: {total_queries}개 쿼리, 건너뜀: {skipped}개")

In [16]:
for query_img in query_relevants.keys():
    db_id = image_filename_to_db_id(query_img)
    try:
        result = collection.get(ids=[db_id], include=["embeddings"])
        if result["embeddings"] is None or len(result["embeddings"]) == 0:
            print(f"[임베딩 없음] {query_img} → {db_id}")
    except Exception as e:
        print(f"[ID 없음] {query_img} → {db_id} | {e}")


[임베딩 없음] 3020150029545-api_xml-0_11.jpg → 3020150029545-api_xml-0-IMG-11
[임베딩 없음] 3020140037690-api_xml-0_11.jpg → 3020140037690-api_xml-0-IMG-11


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

for r in results_table:
    fig, axes = plt.subplots(1, 6, figsize=(24, 4))

    # 쿼리 이미지
    q_path = os.path.join(IMAGE_DIR, r["query"])
    try:
        axes[0].imshow(Image.open(q_path))
    except:
        axes[0].text(0.5, 0.5, "로드실패", ha='center', va='center')
    axes[0].set_title(f"Query\n{r['query']}", fontsize=7, color='blue')
    axes[0].axis('off')

    # Top-5 결과
    relevant_set = query_relevants[r["query"]]
    for j in range(5):
        ax = axes[j + 1]
        if j < len(r["top_retrieved"]):
            fname = r["top_retrieved"][j]
            dist  = r["top_dist"][j] if j < len(r["top_dist"]) else None
            is_hit = fname in relevant_set

            img_path = os.path.join(IMAGE_DIR, fname)
            try:
                ax.imshow(Image.open(img_path))
            except:
                ax.text(0.5, 0.5, "로드실패", ha='center', va='center')

            color = 'green' if is_hit else 'red'
            label = "HIT" if is_hit else "MISS"
            dist_str = f"\ndist={dist:.4f}" if dist else ""
            ax.set_title(f"Top-{j+1} [{label}]{dist_str}\n{fname}", fontsize=6, color=color)
        else:
            ax.text(0.5, 0.5, "없음", ha='center', va='center')
            ax.set_title(f"Top-{j+1}", fontsize=6)
        ax.axis('off')

    hit_str = " | ".join(f"Hit@{k}={r[f'hit@{k}']}" for k in K_LIST)
    fig.suptitle(f"{r['type']} | rank={r['first_hit_rank']} | {hit_str}", fontsize=9, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 4. 쿼리별 상세 결과

### 평가 지표 설명

| 컬럼 | 의미 |
|------|------|
| `query` | 검색에 사용한 쿼리 이미지 |
| `type` | 유사도 유형 |
| `n_relevant` | 해당 쿼리의 정답 이미지 수 |
| `first_hit_rank` | 정답이 처음 등장한 순위 (없으면 miss) |
| `hit@K` | Top-K 안에 정답이 있으면 1, 없으면 0 |

In [ ]:
df = pd.DataFrame(results_table)
df_display = df[["query", "type", "n_relevant", "first_hit_rank"] + [f"hit@{k}" for k in K_LIST]].copy()
df_display

## 5. 요약 테이블 (type별 + 전체)

In [ ]:
type_stats = defaultdict(lambda: {"total": 0, **{k: 0 for k in K_LIST}})
for r in results_table:
    ts = type_stats[r["type"]]
    ts["total"] += 1
    for k in K_LIST:
        ts[k] += r[f"hit@{k}"]

summary_rows = []
for typ, s in type_stats.items():
    summary_rows.append({
        "구분": typ,
        "쿼리수": s["total"],
        **{f"HitRate@{k}": round(s[k] / s["total"], 4) if s["total"] else 0 for k in K_LIST},
    })

summary_rows.append({
    "구분": "전체",
    "쿼리수": total_queries,
    **{f"HitRate@{k}": round(hit_sums[k] / total_queries, 4) if total_queries else 0 for k in K_LIST},
})

df_summary = pd.DataFrame(summary_rows)
df_summary